In [ ]:
import sys
sys.path.append('../Model_info')
import os
import Stdinput as orctm
import netCDF4 as nc
import numpy as np
import time


In [ ]:
INI=orctm.Orctm_cvcm('ORCTM_cvcm','Surface fields for Temperature')
latp_grid = INI.model_info_p('latp_grid')
lonp_grid = INI.model_info_p('lonp_grid')
mask  = INI.model_info_p('weto_p')
mask  = mask.astype(int)

In [ ]:
#cmems_file = './2011/CMEMS-For-CVCM_SUR_file_2011.nc'
#cmems_file = './2012/CMEMS-For-CVCM_SUR_file_2012.nc'
#cmems_file = './2013/CMEMS-For-CVCM_SUR_file_2013.nc'
#cmems_file = './2014/CMEMS-For-CVCM_SUR_file_2014.nc'
#cmems_file = './2015/CMEMS-For-CVCM_SUR_file_2015.nc'
cmems_file = './2016/CMEMS-For-CVCM_SUR_file_2016.nc'
ini_file = nc.Dataset(cmems_file,mode='r',format='netCDF4')
print(ini_file.variables)

latitude = ini_file.variables['latitude'][:]
longitude = ini_file.variables['longitude'][:]
longitude_grid,latitude_grid = np.meshgrid(longitude,latitude)

DEP = ini_file.variables['depth'][:]
to = ini_file.variables['thetao'][:,:,:,:]

to = to.squeeze()
to = to.filled(fill_value = np.nan)
nc.Dataset.close(ini_file)

In [ ]:
start_time = time.time()
# Start the timer
te = 12
to_sst = np.zeros((INI.ie_g,INI.je_g))
Lat_grid = INI.rearrange_grid_to_orctm(latitude_grid)
Lon_grid = INI.rearrange_grid_to_orctm(longitude_grid)
#
import struct
# 
if22 = 2  # Temperature for SST

with open('SURTEM', 'wb') as surtem:
    for t in range(te):
        print('Month: ',str(t+1),' ')
        to_grid  = INI.rearrange_grid_to_orctm(to[t,:,:])

        land = np.zeros_like(to_grid)
        land[np.isnan(to_grid)]=1   #  land 1  Sea: 0 !!!!

        if np.sum(land) != np.size(land):
            to_sst = INI.Grid_bilinear_interpolation( \
                to_grid,land,Lat_grid,Lon_grid,latp_grid,lonp_grid,landvalue=np.nanmean(to_grid))


        to_sst[mask[:,:,0]==0] = np.nan

        # Minimum
        to_sst = np.where(to_sst <= np.nanmin(to_grid),np.nanmin(to_grid),to_sst)
        # Maximum 
        to_sst = np.where(to_sst >= np.nanmax(to_grid),np.nanmax(to_grid),to_sst)

        to_sst[np.isnan(to_sst)==True]=0
        #
        header = struct.pack('>qqqq', int(t+1), int(if22), int(INI.ZZOUT[0]), int(INI.ie_g * INI.je_g))
        header_len = len(header)

        surtem.write(struct.pack(">i", header_len))   
        surtem.write(header)                          
        surtem.write(struct.pack(">i", header_len))  

        tem_sst = to_sst.astype('>f8')         
        tem_sst = tem_sst.T                    
        tem_data = tem_sst.tobytes()           

        tem_len = len(tem_data)
        surtem.write(struct.pack(">i", tem_len))   
        surtem.write(tem_data)                     
        surtem.write(struct.pack(">i", tem_len))   

# Print elapsed time
end_time = time.time()
print(f"Elapsed time: {end_time - start_time:.6f} seconds")